# Capa de datos - Warehouse DuckDB

Esta capa organiza los datos en un modelo dimensional para consultas OLAP. Sigue la idea de la actividad de mini cubo: esquema estrella, tabla de hechos, dimensiones y operaciones OLAP como roll-up, drill-down, slice/dice, pivot, CUBE, ROLLUP y GROUPING SETS.


In [ ]:
from __future__ import annotations

from pathlib import Path
import time

import duckdb
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 80)


## 1. Cargar datos procesados

La capa de datos consume la salida de la capa de analisis: `data/processed/kepler_koi_processed.csv` y `data/processed/pscomppars_processed.csv`. Si esos archivos no existen, primero se debe ejecutar `01_analisis_eda_preprocesamiento.ipynb`.


In [ ]:
CANDIDATE_DATA_DIRS = [Path("data"), Path("mineria") / "data"]
DATA_DIR = next(
    (
        data_dir
        for data_dir in CANDIDATE_DATA_DIRS
        if (data_dir / "processed" / "kepler_koi_processed.csv").exists()
        and (data_dir / "processed" / "pscomppars_processed.csv").exists()
    ),
    None,
)

if DATA_DIR is None:
    raise FileNotFoundError(
        "No se encontraron los CSV procesados. Ejecuta primero 01_analisis_eda_preprocesamiento.ipynb."
    )

PROCESSED_DIR = DATA_DIR / "processed"
kepler = pd.read_csv(PROCESSED_DIR / "kepler_koi_processed.csv")
pscomppars = pd.read_csv(PROCESSED_DIR / "pscomppars_processed.csv")

for df in (kepler, pscomppars):
    for col in df.select_dtypes(include=["object", "string"]).columns:
        df[col] = df[col].astype("string").str.strip().replace({"": pd.NA})

print("Datos procesados cargados desde:", PROCESSED_DIR)
print("Kepler:", kepler.shape)
print("PSCompPars:", pscomppars.shape)


## 2. Diseno dimensional

El modelo usa dos hechos:

- `fact_koi_observations`: senales/candidatos Kepler.
- `fact_confirmed_planets`: exoplanetas confirmados de referencia.

Y dimensiones descriptivas:

- `dim_star`: caracteristicas de la estrella observada.
- `dim_candidate`: identificadores del candidato.
- `dim_disposition`: estado del candidato.
- `dim_sky_position`: coordenadas y bins del cielo.
- `dim_discovery_method`: metodo/facilidad de descubrimiento.
- `dim_discovery_year`: anio de descubrimiento.


In [ ]:
def band_temperature(value: float) -> str:
    if pd.isna(value):
        return "Desconocida"
    if value < 3700:
        return "Fria"
    if value < 5200:
        return "Templada"
    if value < 6500:
        return "Solar"
    return "Caliente"


def band_radius(value: float) -> str:
    if pd.isna(value):
        return "Desconocido"
    if value < 1.25:
        return "Tipo Tierra"
    if value < 2.0:
        return "Super Tierra"
    if value < 6.0:
        return "Sub Neptuno"
    if value < 15.0:
        return "Gigante"
    return "Muy grande"


def sky_bin(value: float, size: int) -> str:
    if pd.isna(value):
        return "Desconocido"
    low = int(np.floor(value / size) * size)
    high = low + size
    return f"{low}-{high}"

koi_base = kepler.copy()
koi_base["star_temp_band"] = koi_base["koi_steff"].apply(band_temperature)
koi_base["planet_radius_band"] = koi_base["koi_prad"].apply(band_radius)
koi_base["ra_bin"] = koi_base["ra"].apply(lambda x: sky_bin(x, 30))
koi_base["dec_bin"] = koi_base["dec"].apply(lambda x: sky_bin(x + 90, 30))

planet_base = pscomppars.copy()
planet_base["star_temp_band"] = planet_base["st_teff"].apply(band_temperature)
planet_base["planet_radius_band"] = planet_base["pl_rade"].apply(band_radius)
planet_base["ra_bin"] = planet_base["ra"].apply(lambda x: sky_bin(x, 30))
planet_base["dec_bin"] = planet_base["dec"].apply(lambda x: sky_bin(x + 90, 30))

# Dimensiones Kepler.
dim_star = koi_base[["kepid", "koi_steff", "koi_slogg", "koi_srad", "koi_kepmag", "star_temp_band"]].drop_duplicates("kepid").copy()
dim_star = dim_star.rename(columns={"kepid": "star_id"})
dim_star.insert(0, "star_key", range(1, len(dim_star) + 1))

dim_candidate = koi_base[["kepoi_name", "kepler_name", "planet_radius_band"]].drop_duplicates("kepoi_name").copy()
dim_candidate = dim_candidate.rename(columns={"kepoi_name": "candidate_id"})
dim_candidate.insert(0, "candidate_key", range(1, len(dim_candidate) + 1))

dim_disposition = pd.DataFrame({"koi_disposition": sorted(koi_base["koi_disposition"].dropna().unique())})
dim_disposition.insert(0, "disposition_key", range(1, len(dim_disposition) + 1))

dim_sky_position = koi_base[["ra", "dec", "ra_bin", "dec_bin"]].drop_duplicates().copy()
dim_sky_position.insert(0, "sky_key", range(1, len(dim_sky_position) + 1))

fact_koi = koi_base.merge(dim_star[["star_key", "star_id"]], left_on="kepid", right_on="star_id", how="left")
fact_koi = fact_koi.merge(dim_candidate[["candidate_key", "candidate_id"]], left_on="kepoi_name", right_on="candidate_id", how="left")
fact_koi = fact_koi.merge(dim_disposition, on="koi_disposition", how="left")
fact_koi = fact_koi.merge(dim_sky_position, on=["ra", "dec", "ra_bin", "dec_bin"], how="left")
fact_koi = fact_koi[[
    "candidate_key", "star_key", "disposition_key", "sky_key",
    "koi_period", "koi_impact", "koi_duration", "koi_depth", "koi_prad", "koi_teq", "koi_insol", "koi_model_snr",
]].rename(columns={
    "koi_period": "orbital_period_days",
    "koi_impact": "impact_parameter",
    "koi_duration": "transit_duration_hours",
    "koi_depth": "transit_depth_ppm",
    "koi_prad": "planet_radius_earth",
    "koi_teq": "equilibrium_temp_k",
    "koi_insol": "insolation_flux",
    "koi_model_snr": "model_snr",
})
fact_koi.insert(0, "observation_key", range(1, len(fact_koi) + 1))

# Dimensiones y hecho de planetas confirmados.
dim_discovery_method = planet_base[["discoverymethod", "disc_facility"]].fillna("Desconocido").drop_duplicates().copy()
dim_discovery_method.insert(0, "method_key", range(1, len(dim_discovery_method) + 1))

dim_discovery_year = planet_base[["disc_year"]].dropna().drop_duplicates().sort_values("disc_year").copy()
dim_discovery_year["disc_year"] = dim_discovery_year["disc_year"].astype(int)
dim_discovery_year.insert(0, "year_key", range(1, len(dim_discovery_year) + 1))

fact_planets = planet_base.merge(dim_discovery_method, on=["discoverymethod", "disc_facility"], how="left")
fact_planets["disc_year_int"] = fact_planets["disc_year"].astype("Int64")
fact_planets = fact_planets.merge(dim_discovery_year, left_on="disc_year_int", right_on="disc_year", how="left")
fact_planets = fact_planets[[
    "pl_name", "hostname", "method_key", "year_key", "pl_orbper", "pl_rade", "pl_bmasse",
    "pl_eqt", "st_teff", "st_rad", "st_mass", "sy_dist", "ra", "dec",
]].copy()
fact_planets.insert(0, "planet_fact_key", range(1, len(fact_planets) + 1))

print(dim_star.shape, dim_candidate.shape, dim_disposition.shape, dim_sky_position.shape, fact_koi.shape)
print(dim_discovery_method.shape, dim_discovery_year.shape, fact_planets.shape)


## 3. Crear warehouse en DuckDB

Se crea `data/warehouse/exoplanets.duckdb`. Las tablas son fisicas y despues se crea una vista para consultar el cubo con joins.


In [ ]:
WAREHOUSE_DIR = DATA_DIR / "warehouse"
WAREHOUSE_DIR.mkdir(exist_ok=True)
DB_PATH = WAREHOUSE_DIR / "exoplanets.duckdb"

con = duckdb.connect(str(DB_PATH))

for name, df in {
    "dim_star": dim_star,
    "dim_candidate": dim_candidate,
    "dim_disposition": dim_disposition,
    "dim_sky_position": dim_sky_position,
    "fact_koi_observations": fact_koi,
    "dim_discovery_method": dim_discovery_method,
    "dim_discovery_year": dim_discovery_year,
    "fact_confirmed_planets": fact_planets,
}.items():
    con.register(f"tmp_{name}", df)
    con.execute(f"CREATE OR REPLACE TABLE {name} AS SELECT * FROM tmp_{name}")
    con.unregister(f"tmp_{name}")

con.execute("""
CREATE OR REPLACE VIEW v_koi_observations AS
SELECT
    f.observation_key,
    c.candidate_id,
    c.kepler_name,
    c.planet_radius_band,
    s.star_id,
    s.star_temp_band,
    s.koi_steff,
    s.koi_slogg,
    s.koi_srad,
    s.koi_kepmag,
    d.koi_disposition,
    sky.ra,
    sky.dec,
    sky.ra_bin,
    sky.dec_bin,
    f.orbital_period_days,
    f.impact_parameter,
    f.transit_duration_hours,
    f.transit_depth_ppm,
    f.planet_radius_earth,
    f.equilibrium_temp_k,
    f.insolation_flux,
    f.model_snr
FROM fact_koi_observations f
JOIN dim_candidate c USING (candidate_key)
JOIN dim_star s USING (star_key)
JOIN dim_disposition d USING (disposition_key)
JOIN dim_sky_position sky USING (sky_key)
""")

con.execute("""
CREATE OR REPLACE VIEW v_confirmed_planets AS
SELECT
    f.planet_fact_key,
    f.pl_name,
    f.hostname,
    m.discoverymethod,
    m.disc_facility,
    y.disc_year,
    f.pl_orbper,
    f.pl_rade,
    f.pl_bmasse,
    f.pl_eqt,
    f.st_teff,
    f.st_rad,
    f.st_mass,
    f.sy_dist,
    f.ra,
    f.dec
FROM fact_confirmed_planets f
LEFT JOIN dim_discovery_method m USING (method_key)
LEFT JOIN dim_discovery_year y USING (year_key)
""")

con.execute("SHOW TABLES").df()


## 4. Operaciones OLAP

Estas consultas son la parte que despues puede exponerse por API para el frontend.


In [ ]:
# Roll-up: resumen general por disposicion.
rollup_disposition = con.execute("""
SELECT
    koi_disposition,
    COUNT(*) AS n_observations,
    AVG(planet_radius_earth) AS avg_radius_earth,
    MEDIAN(planet_radius_earth) AS median_radius_earth,
    AVG(model_snr) AS avg_snr
FROM v_koi_observations
GROUP BY koi_disposition
ORDER BY n_observations DESC
""").df()
rollup_disposition


In [ ]:
# Drill-down: detalle por disposicion y banda de temperatura estelar.
drilldown_temp = con.execute("""
SELECT
    koi_disposition,
    star_temp_band,
    planet_radius_band,
    COUNT(*) AS n_observations,
    MEDIAN(planet_radius_earth) AS median_radius_earth
FROM v_koi_observations
GROUP BY koi_disposition, star_temp_band, planet_radius_band
ORDER BY koi_disposition, star_temp_band, n_observations DESC
""").df()
drilldown_temp.head(20)


In [ ]:
# Slice & dice: candidatos con temperatura de equilibrio moderada y radio tipo Tierra/Super Tierra.
slice_dice_habitable = con.execute("""
SELECT
    koi_disposition,
    star_temp_band,
    planet_radius_band,
    COUNT(*) AS n_observations,
    MEDIAN(equilibrium_temp_k) AS median_teq
FROM v_koi_observations
WHERE equilibrium_temp_k BETWEEN 180 AND 320
  AND planet_radius_band IN ('Tipo Tierra', 'Super Tierra')
GROUP BY koi_disposition, star_temp_band, planet_radius_band
ORDER BY n_observations DESC
""").df()
slice_dice_habitable


In [ ]:
# Pivot: clases como columnas por banda de temperatura.
pivot_disposition = con.execute("""
PIVOT v_koi_observations
ON koi_disposition
USING COUNT(*)
GROUP BY star_temp_band
ORDER BY star_temp_band
""").df()
pivot_disposition


## 5. CUBE, ROLLUP y GROUPING SETS

Estas consultas materializan cuboides de diferentes niveles, como pide la actividad de DuckDB.


In [ ]:
cube_df = con.execute("""
SELECT
    star_temp_band,
    planet_radius_band,
    koi_disposition,
    COUNT(*) AS n_observations,
    AVG(model_snr) AS avg_snr
FROM v_koi_observations
GROUP BY CUBE (star_temp_band, planet_radius_band, koi_disposition)
ORDER BY n_observations DESC
""").df()

level_counts = con.execute("""
SELECT
    COUNT(DISTINCT star_temp_band) AS l_star_temp,
    COUNT(DISTINCT planet_radius_band) AS l_radius,
    COUNT(DISTINCT koi_disposition) AS l_disposition
FROM v_koi_observations
""").df().iloc[0]

expected_rows = int((level_counts["l_star_temp"] + 1) * (level_counts["l_radius"] + 1) * (level_counts["l_disposition"] + 1))
pd.DataFrame({
    "metrica": ["filas_cube", "filas_esperadas_formula"],
    "valor": [len(cube_df), expected_rows],
})


In [ ]:
rollup_df = con.execute("""
SELECT
    star_temp_band,
    planet_radius_band,
    COUNT(*) AS n_observations,
    AVG(planet_radius_earth) AS avg_radius_earth
FROM v_koi_observations
GROUP BY ROLLUP (star_temp_band, planet_radius_band)
ORDER BY star_temp_band, planet_radius_band
""").df()
rollup_df.head(20)


In [ ]:
grouping_sets_df = con.execute("""
SELECT
    star_temp_band,
    koi_disposition,
    COUNT(*) AS n_observations,
    MEDIAN(planet_radius_earth) AS median_radius_earth
FROM v_koi_observations
GROUP BY GROUPING SETS ((star_temp_band), (koi_disposition), (star_temp_band, koi_disposition), ())
ORDER BY star_temp_band, koi_disposition
""").df()
grouping_sets_df.head(30)


## 6. Medida distributiva vs holistica e iceberg cube

`COUNT`/`SUM` son distributivas porque se pueden combinar por partes. `MEDIAN` es holistica porque requiere conocer la distribucion completa de valores.


In [ ]:
count_query = """
SELECT star_temp_band, planet_radius_band, COUNT(*) AS n_observations
FROM v_koi_observations
GROUP BY GROUPING SETS ((star_temp_band), (planet_radius_band), (star_temp_band, planet_radius_band), ())
"""

median_query = """
SELECT star_temp_band, planet_radius_band, MEDIAN(planet_radius_earth) AS median_radius
FROM v_koi_observations
GROUP BY GROUPING SETS ((star_temp_band), (planet_radius_band), (star_temp_band, planet_radius_band), ())
"""

def medir(query: str, repeticiones: int = 10) -> float:
    inicio = time.perf_counter()
    for _ in range(repeticiones):
        con.execute(query).fetchall()
    return (time.perf_counter() - inicio) / repeticiones

pd.DataFrame({
    "medida": ["COUNT distributiva", "MEDIAN holistica"],
    "segundos_promedio": [medir(count_query), medir(median_query)],
})


In [ ]:
iceberg_cube = con.execute("""
SELECT
    star_temp_band,
    planet_radius_band,
    koi_disposition,
    COUNT(*) AS n_observations,
    AVG(model_snr) AS avg_snr
FROM v_koi_observations
GROUP BY star_temp_band, planet_radius_band, koi_disposition
HAVING COUNT(*) >= 50
ORDER BY n_observations DESC
""").df()
iceberg_cube


In [ ]:
# Cerrar la conexion libera el archivo DuckDB para la siguiente capa.
con.close()
print(f"Warehouse guardado en: {DB_PATH}")
